In [1]:
import tensorflow as tf
from tensorflow.keras.applications import Xception
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D

In [2]:
teacher = tf.keras.models.load_model(
    r"C:\Users\k nithin\Downloads\M.Tech Project\Code\ResNet101_lentil_10.keras",
    compile=False
)

teacher.trainable = False  # 🔒 Freeze teacher completely
print("Teacher loaded and frozen")


Teacher loaded and frozen


In [3]:
print("Trainable variables in teacher:",
      len(teacher.trainable_variables))

Trainable variables in teacher: 0


In [4]:
TEACHER_HINT_LAYERS = [
    "conv4_block23_out",  # deep semantic features
    "conv5_block3_out"    # highest-level features
]

In [5]:
teacher_feature_extractor = Model(
    inputs=teacher.input,
    outputs=[teacher.get_layer(name).output for name in TEACHER_HINT_LAYERS],
    name="teacher_feature_extractor"
)

teacher_feature_extractor.trainable = False

In [6]:
teacher_feature_extractor.summary()

Model: "teacher_feature_extractor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 224, 224, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_pad (ZeroPadding2D)     │ (None, 230, 230, 3)       │               0 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_conv (Conv2D)           │ (None, 112, 112, 64)      │           9,472 │ conv1_pad[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_bn (BatchNormalization) │ (None, 112, 112, 64)      │             256 │ conv1_conv[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_relu (Activation)       │ (None, 112, 112, 64)      │               0 │ conv1_bn[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ pool1_pad (ZeroPadding2D)     │ (None, 114, 114, 64)      │               0 │ conv1_relu[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ pool1_pool (MaxPooling2D)     │ (None, 56, 56, 64)        │               0 │ pool1_pad[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_conv (Conv2D)  │ (None, 56, 56, 64)        │           4,160 │ pool1_pool[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_bn             │ (None, 56, 56, 64)        │             256 │ conv2_block1_1_conv[0][0]  │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_relu           │ (None, 56, 56, 64)        │               0 │ conv2_block1_1_bn[0][0]    │
│ (Activation)                  │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_2_conv (Conv2D)  │ (None, 56, 56, 64)        │          36,928 │ conv2_block1_1_relu[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_2_bn             │ (None, 56, 56, 64)        │             256 │ conv2_block1_2_conv[0][0]  │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_2_relu           │ (None, 56, 56, 64)        │               0 │ conv2_block1_2_bn[0][0]    │
│ (Activation)                  │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_0_conv (Conv2D)  │ (None, 56, 56, 256)       │          16,640 │ pool1_pool[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_3_conv (Conv2D)  │ (None, 56, 56, 256)       │          16,640 │ conv2_block1_2_relu[0][0]  │
├───────────────────────────────┼───────────────────────────┼───────────────

 Total params: 42,658,176 (162.73 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 42,658,176 (162.73 MB)

In [7]:
student_base = Xception(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

In [8]:
STUDENT_HINT_LAYERS = [
    "block13_sepconv2_act",   # mid-deep semantic (~14x14)
    "block14_sepconv2_act"    # final semantic (~7x7)
]

In [9]:
student_feature_extractor = tf.keras.Model(
    inputs=student_base.input,
    outputs=[
        student_base.get_layer(name).output
        for name in STUDENT_HINT_LAYERS
    ]
)

In [10]:
def projection_layer(x, out_channels, name):
    return tf.keras.layers.Conv2D(
        out_channels,
        kernel_size=1,
        padding="same",
        name=name
    )(x)

In [11]:
# Classification head for the student model

x = student_base.output
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(512, activation="relu")(x)

student_logits = tf.keras.layers.Dense(
    3,
    activation=None,
    name="bean_logits"
)(x)

In [12]:
import tensorflow as tf

teacher_channels = [
    teacher_feature_extractor.output[i].shape[-1]
    for i in range(len(teacher_feature_extractor.output))
]

student_features_raw = student_feature_extractor.output
projected_student_features = []

for i, s_feat in enumerate(student_features_raw):

    student_ch = s_feat.shape[-1]
    teacher_ch = teacher_channels[i]

    if student_ch != teacher_ch:
        s_feat = tf.keras.layers.Conv2D(
            teacher_ch,
            kernel_size=1,
            padding="same",
            name=f"proj_{i}"
        )(s_feat)

    projected_student_features.append(s_feat)

In [13]:
student_model = tf.keras.Model(
    inputs=student_base.input,
    outputs={
        "logits": student_logits,
        "features": projected_student_features
    }
)

In [14]:
import tensorflow as tf
dummy = tf.random.normal((2,224,224,3))

teacher_feats = teacher_feature_extractor(dummy)
student_out = student_model(dummy)

student_feats = student_out["features"]   # <-- IMPORTANT

for i, (tf_feat, sf_feat) in enumerate(zip(teacher_feats, student_feats)):
    print(f"\nLayer {i}")
    print("Teacher mean/std:", tf_feat.numpy().mean(), tf_feat.numpy().std())
    print("Student mean/std:", sf_feat.numpy().mean(), sf_feat.numpy().std())


Layer 0
Teacher mean/std: 1.5198412 2.180192
Student mean/std: 0.0017635257 0.2567776

Layer 1
Teacher mean/std: 0.10601478 0.70313156
Student mean/std: 0.08606268 0.28354087


In [15]:
for i, (tf, sf) in enumerate(zip(teacher_feats, student_feats)):
    print(f"\nLayer {i}")
    print("Teacher mean/std:", tf.numpy().mean(), tf.numpy().std())
    print("Student mean/std:", sf.numpy().mean(), sf.numpy().std())


Layer 0
Teacher mean/std: 1.5198412 2.180192
Student mean/std: 0.0017635257 0.2567776

Layer 1
Teacher mean/std: 0.10601478 0.70313156
Student mean/std: 0.08606268 0.28354087


In [16]:
def normalize_feature(f):
    # Normalize per-channel
    mean = tf.reduce_mean(f, axis=[1, 2], keepdims=True)
    std = tf.math.reduce_std(f, axis=[1, 2], keepdims=True) + 1e-6
    return (f - mean) / std

In [17]:
def feature_kd_loss(teacher_feats, student_feats):
    loss = 0.0
    for t_feat, s_feat in zip(teacher_feats, student_feats):
        t_feat = tf.stop_gradient(normalize_feature(t_feat))
        s_feat = normalize_feature(s_feat)
        loss += tf.reduce_mean(tf.square(t_feat - s_feat))
    return loss

In [18]:
NUM_CLASSES = 3  # example: healthy, rust, anthracnose

In [19]:
import tensorflow as tf

In [20]:
classification_loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True
)

In [21]:
KD_WEIGHT = 1.0   # λ

In [22]:
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)

In [23]:
#@tf.function
def train_step(images, labels):
    # 1. Teacher forward (features only)
    teacher_feats = teacher_feature_extractor(images)

    with tf.GradientTape() as tape:
        # 2. Student forward
        outputs = student_model(images, training=True)
        student_logits = outputs["logits"]
        student_feats = outputs["features"]

        # 3. Losses
        cls_loss = classification_loss_fn(labels, student_logits)
        kd_loss = feature_kd_loss(teacher_feats, student_feats)

        total_loss = cls_loss + KD_WEIGHT * kd_loss

    # 4. Backprop (student only)
    grads = tape.gradient(total_loss, student_model.trainable_variables)
    optimizer.apply_gradients(zip(grads, student_model.trainable_variables))

    return {
        "total_loss": total_loss,
        "cls_loss": cls_loss,
        "kd_loss": kd_loss
    }

In [24]:
dummy_y = tf.constant([0, 1])  # fake labels

losses = train_step(dummy_x, dummy_y)
print(losses)

NameError: name 'dummy_x' is not defined

In [25]:
import tensorflow as tf
import pathlib

DATA_DIR = pathlib.Path(r"C:\Users\k nithin\Downloads\M.Tech Project\Beans dataset")
IMG_SIZE = (224, 224)
BATCH_SIZE = 8
SEED = 42
NUM_CLASSES = 3


In [26]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    labels="inferred",
    label_mode="int",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
    validation_split=0.3,
    subset="training"
)

Found 59069 files belonging to 3 classes.
Using 41349 files for training.


In [27]:
val_test_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    labels="inferred",
    label_mode="int",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
    validation_split=0.3,
    subset="validation"
)

Found 59069 files belonging to 3 classes.
Using 17720 files for validation.


In [28]:
val_batches = tf.data.experimental.cardinality(val_test_ds).numpy()
test_ds = val_test_ds.take(val_batches // 2)
val_ds = val_test_ds.skip(val_batches // 2)

In [29]:
class_names = train_ds.class_names
print("Class order:", class_names)

Class order: ['anthra', 'healthy', 'rust']


In [30]:
train_batches = tf.data.experimental.cardinality(train_ds).numpy()
print("Train batches per epoch:", train_batches)
print("Approx images per epoch:", train_batches * BATCH_SIZE)

Train batches per epoch: 5169
Approx images per epoch: 41352


In [31]:
from tensorflow.keras.applications.resnet import preprocess_input

def preprocess(image, label):
    image = tf.cast(image, tf.float32)
    image = preprocess_input(image)
    return image, label

In [32]:
train_ds = train_ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
test_ds = test_ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)

In [33]:
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)

In [34]:
import time

images, labels = next(iter(train_ds))

start = time.time()
losses = train_step(images, labels)
end = time.time()

print("One train_step time (seconds):", end - start)
print(losses)

One train_step time (seconds): 6.2668867111206055
{'total_loss': <tf.Tensor: shape=(), dtype=float32, numpy=4.3364996910095215>, 'cls_loss': <tf.Tensor: shape=(), dtype=float32, numpy=1.031854510307312>, 'kd_loss': <tf.Tensor: shape=(), dtype=float32, numpy=3.304645299911499>}


In [35]:
STEPS_PER_EPOCH = 100
EPOCHS = 10

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    # ---------- TRAINING ----------
    step = 0
    for images, labels in train_ds:
        losses = train_step(images, labels)
        step += 1

        if step >= STEPS_PER_EPOCH:
            break

    print(
        f"Train | total: {losses['total_loss']:.4f} | "
        f"cls: {losses['cls_loss']:.4f} | "
        f"kd: {losses['kd_loss']:.4f}"
    )

    # ---------- VALIDATION ----------
    # ✅ ADD THESE TWO LINES
    val_cls_loss = tf.keras.metrics.Mean()
    val_acc = tf.keras.metrics.SparseCategoricalAccuracy()

    for images, labels in val_ds:
        outputs = student_model(images, training=False)
        logits = outputs["logits"]

        loss = classification_loss_fn(labels, logits)
        val_cls_loss.update_state(loss)
        val_acc.update_state(labels, logits)

    print(
        f"Val   | cls_loss: {val_cls_loss.result():.4f} | "
        f"acc: {val_acc.result():.4f}"
    )



Epoch 1/10
Train | total: 3.3594 | cls: 0.0530 | kd: 3.3064
Val   | cls_loss: 0.1182 | acc: 0.9572

Epoch 2/10
Train | total: 2.9712 | cls: 0.0175 | kd: 2.9538
Val   | cls_loss: 0.1130 | acc: 0.9585

Epoch 3/10
Train | total: 3.5289 | cls: 0.5618 | kd: 2.9671
Val   | cls_loss: 0.0610 | acc: 0.9809

Epoch 4/10
Train | total: 3.3989 | cls: 0.5954 | kd: 2.8035
Val   | cls_loss: 0.0464 | acc: 0.9861

Epoch 5/10
Train | total: 2.6762 | cls: 0.0081 | kd: 2.6680
Val   | cls_loss: 0.0440 | acc: 0.9874

Epoch 6/10
Train | total: 2.6116 | cls: 0.0048 | kd: 2.6068
Val   | cls_loss: 0.0324 | acc: 0.9914

Epoch 7/10
Train | total: 2.5939 | cls: 0.0556 | kd: 2.5382
Val   | cls_loss: 0.0270 | acc: 0.9939

Epoch 8/10
Train | total: 2.3976 | cls: 0.1486 | kd: 2.2490
Val   | cls_loss: 0.0354 | acc: 0.9919

Epoch 9/10
Train | total: 2.2289 | cls: 0.0101 | kd: 2.2188
Val   | cls_loss: 0.0372 | acc: 0.9894

Epoch 10/10
Train | total: 2.3419 | cls: 0.1321 | kd: 2.2099
Val   | cls_loss: 0.0270 | acc: 0.9931